In [3]:
import requests
r = requests.get("http://localhost:11434/api/tags")
print("Ollama is running!", r.json())

Ollama is running! {'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-06-12T07:11:10.311803471Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768}, 'capabilities': ['embedding']}, {'name': 'llama3.2:latest', 'model': 'llama3.2:latest', 'modified_at': '2026-06-12T07:10:23.087497531Z', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}]}


In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./Docs/Bhaubali.pdf")
doc = loader.load()   
print(doc)

C:\Users\singh\AppData\Local\Temp\ipykernel_22244\478108686.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-13T16:56:52+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-13T16:56:52+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './Docs/Bhaubali.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1'}, page_content='Bhabhuli: The Girl Who Followed the Wind'), Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-13T16:56:52+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-13T16:56:52+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './Docs/Bhaubali.pdf', 'total_pages': 51, 'page': 1, 'page_label': '2'}, page_content='Chapter 1\nBhabhuli lived in a small village surrounded by fields, rivers, and ancient trees. In this chapter, she\nfaced a new challenge and learned an important le

Creating own metadata


In [9]:
for i in doc:
    i.metadata={"Source":"PDF","Movie":"Bhaubali"}

Splitting docs into chunks


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)
chunk=splitter.split_documents(doc)
chunk

[Document(metadata={'Source': 'PDF', 'Movie': 'Bhaubali'}, page_content='Bhabhuli: The Girl Who Followed the Wind'),
 Document(metadata={'Source': 'PDF', 'Movie': 'Bhaubali'}, page_content='Chapter 1\nBhabhuli lived in a small village surrounded by fields, rivers, and ancient trees. In this chapter, she'),
 Document(metadata={'Source': 'PDF', 'Movie': 'Bhaubali'}, page_content='faced a new challenge and learned an important lesson about courage, friendship, kindness, and\ndetermination. As she followed the mysterious wind that seemed to guide her path, she met'),
 Document(metadata={'Source': 'PDF', 'Movie': 'Bhaubali'}, page_content='travelers, helped villagers, solved problems, and discovered hidden stories of the land. Each\nexperience made her wiser and stronger. By the end of the chapter, Bhabhuli realized that true'),
 Document(metadata={'Source': 'PDF', 'Movie': 'Bhaubali'}, page_content='strength comes not from power but from helping others and staying true to oneself. The wind

In [13]:
chunk[0].metadata

{'Source': 'PDF', 'Movie': 'Bhaubali'}

In [14]:
len(chunk)

551

Creat Embedding


In [18]:
from langchain_ollama import  OllamaEmbeddings,OllamaLLM
from langchain_community.vectorstores import Chroma
embedding_model = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")
vectorstore=Chroma.from_documents(documents=doc,embedding=embedding_model,persist_directory="./Vector/")
print("Embeddings created and stored!")


Embeddings created and stored!


In [17]:
vectorstore.similarity_search("explain the script?",k=3)

[Document(metadata={'total_pages': 51, 'producer': 'ReportLab PDF Library - (opensource)', 'title': '(anonymous)', 'subject': '(unspecified)', 'author': '(anonymous)', 'creationdate': '2026-06-13T16:56:52+00:00', 'keywords': '', 'page_label': '1', 'trapped': '/False', 'page': 0, 'source': './Docs/Bhaubali.pdf', 'moddate': '2026-06-13T16:56:52+00:00', 'creator': '(unspecified)'}, page_content='Bhabhuli: The Girl Who Followed the Wind'),
 Document(metadata={'Source': 'PDF', 'Movie': 'Bhaubali'}, page_content='Bhabhuli: The Girl Who Followed the Wind'),
 Document(metadata={'page_label': '44', 'producer': 'ReportLab PDF Library - (opensource)', 'trapped': '/False', 'page': 43, 'title': '(anonymous)', 'source': './Docs/Bhaubali.pdf', 'total_pages': 51, 'subject': '(unspecified)', 'author': '(anonymous)', 'creationdate': '2026-06-13T16:56:52+00:00', 'creator': '(unspecified)', 'keywords': '', 'moddate': '2026-06-13T16:56:52+00:00'}, page_content='Chapter 43\nBhabhuli lived in a small village

Talk to LLM


In [20]:

llm = OllamaLLM(base_url="http://localhost:11434", model="llama3.2")
context=vectorstore.similarity_search("Bhaubali",k=3)
response=llm.invoke("Story of bhaubali movie?:{context}")
print(response)

Bhaag Baag Banay (not Bhaubali) is a Hindi-language Indian television series that aired from 2009 to 2012. However, I think you are referring to the famous Telugu film "Baahubali" or its sequel.

The story of "Baahubali: The Beginning" and its sequel "Baahubali 2: The Conclusion" is as follows:

In the kingdom of Mahishmati, a powerful and prosperous empire ruled by the Maharaja Bhalladeva (played by Rana Daggubati) and his brother Mahendra (played by Prabhas). The two brothers were known for their bravery and loyalty to each other.

The story begins with the death of their father, Indrajeet, a wise and just king who had made a promise to sacrifice himself to save the kingdom from the demon Mahishasura. Bhalladeva's brother, Mahendra, has gone missing while on a journey to find a magical stone called Ammanathagudi.

Meanwhile, in a small village, a young man named Avanthika (played by Anushka Shetty) lives with her husband, Arjun, and their son, Baahubali. After the death of Indrajeet,